# Assignment 1 — Python Fundamentals for Data Streaming
## AITU Campus Shuttle and Mobility Data

**Student:** Tulebay Aslan  
**Group:** BDA-2403
**Student ID:** [241748]  
**Instructor:** [Serek Azamat]  
**Submission date:** 14.09.2026  
**Environment:** Google Colab  
**Git repository / project link:** [https://github.com/Asslan77/Assignment_1_Asslan77]  
**Python version:** Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

In [5]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## Introduction

This assignment focuses on the basic principles of processing streaming data using Python.
A synthetic dataset representing AITU campus shuttle events is used for the analysis.

Each event contains information about the timestamp, route, bus, number of passengers,
speed and current status of the bus.

The main objectives of this assignment are:
- to represent and validate streaming events;
- to parse timestamps and numeric values;
- to implement a generator for sequential event processing;
- to simulate a real-time data stream;
- to convert events to JSON;
- to implement a FastAPI REST endpoint;
- to calculate passenger statistics;
- to create and execute test cases.

In [6]:
!pip install fastapi httpx pandas pytest -q

import pandas as pd
import json

from datetime import datetime
from typing import Dict, Any, Generator, List, Tuple

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

print("Environment and dependencies successfully initialized!")

Environment and dependencies successfully initialized!


# Part A — Data Structures and Parsing

The shuttle event stream is represented as a list of Python dictionaries.
Each dictionary contains information about one shuttle event.

The dataset includes the timestamp, route, bus identifier, passenger count,
speed and current status.

The input data is validated before processing. Timestamps are parsed using
the HH:MM format, while passenger count and speed are checked according to
the required validation rules.

In [8]:
SAMPLE_DATA = [
    {"Timestamp": "08:00", "Route": "AITU-Campus-Residence", "Bus": "B01", "Passengers": 18, "Speed_kmh": 31, "Status": "ON_ROUTE"},
    {"Timestamp": "08:01", "Route": "AITU-Campus-Residence", "Bus": "B02", "Passengers": 22, "Speed_kmh": 28, "Status": "ON_ROUTE"},
    {"Timestamp": "08:02", "Route": "AITU-Campus-Residence", "Bus": "B01", "Passengers": 21, "Speed_kmh": 29, "Status": "ON_ROUTE"},
    {"Timestamp": "08:03", "Route": "AITU-Campus-Residence", "Bus": "B02", "Passengers": 25, "Speed_kmh": 27, "Status": "ON_ROUTE"},
    {"Timestamp": "08:04", "Route": "AITU-Campus-Residence", "Bus": "B01", "Passengers": 24, "Speed_kmh": 0, "Status": "STOPPED"},
    {"Timestamp": "08:05", "Route": "AITU-Campus-Residence", "Bus": "B02", "Passengers": 26, "Speed_kmh": 30, "Status": "ON_ROUTE"},
    {"Timestamp": "08:06", "Route": "AITU-Campus-Residence", "Bus": "B01", "Passengers": 23, "Speed_kmh": 32, "Status": "ON_ROUTE"},
    {"Timestamp": "08:07", "Route": "AITU-Campus-Residence", "Bus": "B02", "Passengers": 28, "Speed_kmh": 26, "Status": "ON_ROUTE"},
    {"Timestamp": "08:08", "Route": "AITU-Campus-Residence", "Bus": "B01", "Passengers": 27, "Speed_kmh": 25, "Status": "ON_ROUTE"},
    {"Timestamp": "08:09", "Route": "AITU-Campus-Residence", "Bus": "B02", "Passengers": 30, "Speed_kmh": 24, "Status": "ON_ROUTE"},
]

print("Number of events:", len(SAMPLE_DATA))
print("\nFirst event:")
print(SAMPLE_DATA[0])

Number of events: 10

First event:
{'Timestamp': '08:00', 'Route': 'AITU-Campus-Residence', 'Bus': 'B01', 'Passengers': 18, 'Speed_kmh': 31, 'Status': 'ON_ROUTE'}


## Event Validation

The `validate_and_parse_event()` function validates one incoming shuttle event.

The validation rules are:

- Timestamp must use the HH:MM format.
- Passengers must be a non-negative integer.
- Speed_kmh must be between 0 and 120 km/h.
- Status must be either ON_ROUTE or STOPPED.

The function returns whether the event is valid, a list of validation errors,
and the parsed event.

In [10]:
def validate_and_parse_event(
    event: Dict[str, Any]
) -> Tuple[bool, List[str], Dict[str, Any]]:

    errors = []
    parsed_event = event.copy()


    try:
        parsed_event["Timestamp_parsed"] = datetime.strptime(
            event.get("Timestamp", ""),
            "%H:%M"
        ).time()
    except (ValueError, TypeError):
        errors.append(
            "Invalid Timestamp format. Expected HH:MM."
        )

    passengers = event.get("Passengers")

    if not isinstance(passengers, int) or isinstance(passengers, bool) or passengers < 0:
        errors.append(
            "Passengers must be a non-negative integer."
        )

    speed = event.get("Speed_kmh")

    if (
        not isinstance(speed, (int, float))
        or isinstance(speed, bool)
        or not (0 <= speed <= 120)
    ):
        errors.append(
            "Speed_kmh must be between 0 and 120."
        )

    status = event.get("Status")

    if status not in ["ON_ROUTE", "STOPPED"]:
        errors.append(
            "Status must be either 'ON_ROUTE' or 'STOPPED'."
        )

    is_valid = len(errors) == 0

    return is_valid, errors, parsed_event

In [11]:
print("Validation results:\n")

for event in SAMPLE_DATA:
    is_valid, errors, parsed = validate_and_parse_event(event)

    if is_valid:
        print(f"{event['Timestamp']} -> VALID")
    else:
        print(f"{event['Timestamp']} -> INVALID")
        print("Errors:", errors)

Validation results:

08:00 -> VALID
08:01 -> VALID
08:02 -> VALID
08:03 -> VALID
08:04 -> VALID
08:05 -> VALID
08:06 -> VALID
08:07 -> VALID
08:08 -> VALID
08:09 -> VALID


### Data Types and Validation Rules

| Field | Python Type | Validation Rule | Example |
|---|---|---|---|
| Timestamp | `str` / parsed `datetime.time` | HH:MM format | `08:00` |
| Passengers | `int` | Non-negative integer | `18` |
| Speed_kmh | `int` / `float` | 0–120 km/h | `31` |
| Status | `str` | ON_ROUTE or STOPPED | `ON_ROUTE` |

# Part B — Streaming Simulation

A generator is used to simulate the arrival of shuttle events in chronological order.

The `event_stream_generator()` function uses the `yield` statement to return
one event at a time. This approach avoids creating another complete collection
inside the processing function and is suitable for large or unlimited streams.

In [12]:
def event_stream_generator(
    data: List[Dict[str, Any]]
) -> Generator[Dict[str, Any], None, None]:

    for event in data:
        is_valid, errors, parsed_event = validate_and_parse_event(event)

        if is_valid:
            yield parsed_event

In [13]:
print("Streaming events:\n")

stream = event_stream_generator(SAMPLE_DATA)

for event in stream:
    print(
        f"{event['Timestamp']} | "
        f"{event['Bus']} | "
        f"Passengers: {event['Passengers']} | "
        f"Speed: {event['Speed_kmh']} | "
        f"Status: {event['Status']}"
    )

Streaming events:

08:00 | B01 | Passengers: 18 | Speed: 31 | Status: ON_ROUTE
08:01 | B02 | Passengers: 22 | Speed: 28 | Status: ON_ROUTE
08:02 | B01 | Passengers: 21 | Speed: 29 | Status: ON_ROUTE
08:03 | B02 | Passengers: 25 | Speed: 27 | Status: ON_ROUTE
08:04 | B01 | Passengers: 24 | Speed: 0 | Status: STOPPED
08:05 | B02 | Passengers: 26 | Speed: 30 | Status: ON_ROUTE
08:06 | B01 | Passengers: 23 | Speed: 32 | Status: ON_ROUTE
08:07 | B02 | Passengers: 28 | Speed: 26 | Status: ON_ROUTE
08:08 | B01 | Passengers: 27 | Speed: 25 | Status: ON_ROUTE
08:09 | B02 | Passengers: 30 | Speed: 24 | Status: ON_ROUTE


## Streaming Simulation Results

| Event | Passengers | Speed | Status | Processed? | Reason |
|---:|---:|---:|---|---|---|
| 1 | 18 | 31 | ON_ROUTE | Yes | Valid event |
| 2 | 22 | 28 | ON_ROUTE | Yes | Valid event |
| 3 | 21 | 29 | ON_ROUTE | Yes | Valid event |
| 4 | 25 | 27 | ON_ROUTE | Yes | Valid event |
| 5 | 24 | 0 | STOPPED | Yes | Valid stopped event |
| 6 | 26 | 30 | ON_ROUTE | Yes | Valid event |
| 7 | 23 | 32 | ON_ROUTE | Yes | Valid event |
| 8 | 28 | 26 | ON_ROUTE | Yes | Valid event |
| 9 | 27 | 25 | ON_ROUTE | Yes | Valid event |
| 10 | 30 | 24 | ON_ROUTE | Yes | Valid event |

All ten supplied events are valid and therefore are processed by the generator.
The event at 08:04 is valid even though the speed is 0 because zero is included
in the allowed speed range.

# Part C — JSON and REST API Processing

The shuttle events can be converted from Python dictionaries into JSON format.
JSON provides a standard format for transferring event data between applications
and REST API services.

In [14]:
json_sample = json.dumps(SAMPLE_DATA[0], indent=4)

print(json_sample)

{
    "Timestamp": "08:00",
    "Route": "AITU-Campus-Residence",
    "Bus": "B01",
    "Passengers": 18,
    "Speed_kmh": 31,
    "Status": "ON_ROUTE"
}


## Occupancy Categories

Passenger counts are converted into occupancy categories using the following rules:

| Passenger Count | Occupancy Category |
|---:|---|
| 0–10 | LOW |
| 11–20 | MEDIUM |
| 21–30 | HIGH |
| >30 | OVER CAPACITY |

In [18]:
def get_occupancy_category(passengers: int) -> str:
    """Categorizes bus passenger load."""

    if passengers <= 10:
        return "LOW"
    elif passengers <= 20:
        return "MEDIUM"
    elif passengers <= 30:
        return "HIGH"
    else:
        return "OVER CAPACITY"

In [19]:
test_passengers = [5, 18, 25, 31]

for passengers in test_passengers:
    print(
        f"{passengers} passengers -> "
        f"{get_occupancy_category(passengers)}"
    )

5 passengers -> LOW
18 passengers -> MEDIUM
25 passengers -> HIGH
31 passengers -> OVER CAPACITY


## FastAPI REST API

A FastAPI application is implemented with a `POST /events` endpoint.

The endpoint accepts one shuttle event, validates its fields and calculates
the occupancy category.

The API returns:
- accepted or rejected status;
- validation errors;
- occupancy category.

In [22]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="AITU Campus Shuttle Streaming Service")


class ShuttleEventSchema(BaseModel):
    Timestamp: str = Field(
        ...,
        json_schema_extra={"example": "08:00"}
    )
    Route: str = Field(
        ...,
        json_schema_extra={"example": "AITU-Campus-Residence"}
    )
    Bus: str = Field(
        ...,
        json_schema_extra={"example": "B01"}
    )
    Passengers: int = Field(
        ...,
        json_schema_extra={"example": 18}
    )
    Speed_kmh: float = Field(
        ...,
        json_schema_extra={"example": 31}
    )
    Status: str = Field(
        ...,
        json_schema_extra={"example": "ON_ROUTE"}
    )


@app.post("/events")
def process_event_endpoint(event: ShuttleEventSchema):
    event_dict = event.model_dump()

    is_valid, errors, _ = validate_and_parse_event(event_dict)

    if not is_valid:
        return {
            "status": "rejected",
            "errors": errors,
            "occupancy_category": None
        }

    category = get_occupancy_category(event_dict["Passengers"])

    return {
        "status": "accepted",
        "errors": [],
        "occupancy_category": category
    }

In [25]:
client = TestClient(app)

valid_payload = {
    "Timestamp": "08:00",
    "Route": "AITU-Campus-Residence",
    "Bus": "B01",
    "Passengers": 18,
    "Speed_kmh": 31,
    "Status": "ON_ROUTE"
}

res1 = client.post("/events", json=valid_payload)

print("Test 1 - Valid Event:")
print(res1.json())


invalid_payload = {
    "Timestamp": "08:04",
    "Route": "AITU-Campus-Residence",
    "Bus": "B01",
    "Passengers": 24,
    "Speed_kmh": 150,
    "Status": "ON_ROUTE"
}

res2 = client.post("/events", json=invalid_payload)

print("\nTest 2 - Invalid Speed:")
print(res2.json())


over_capacity_payload = {
    "Timestamp": "08:10",
    "Route": "AITU-Campus-Residence",
    "Bus": "B01",
    "Passengers": 35,
    "Speed_kmh": 20,
    "Status": "ON_ROUTE"
}

res3 = client.post("/events", json=over_capacity_payload)

print("\nTest 3 - Over Capacity:")
print(res3.json())


invalid_status_payload = {
    "Timestamp": "08:11",
    "Route": "AITU-Campus-Residence",
    "Bus": "B02",
    "Passengers": 20,
    "Speed_kmh": 25,
    "Status": "UNKNOWN"
}

res4 = client.post("/events", json=invalid_status_payload)

print("\nTest 4 - Invalid Status:")
print(res4.json())

Test 1 - Valid Event:
{'status': 'accepted', 'errors': [], 'occupancy_category': 'MEDIUM'}

Test 2 - Invalid Speed:
{'status': 'rejected', 'errors': ['Speed_kmh must be between 0 and 120.'], 'occupancy_category': None}

Test 3 - Over Capacity:
{'status': 'accepted', 'errors': [], 'occupancy_category': 'OVER CAPACITY'}

Test 4 - Invalid Status:
{'status': 'rejected', 'errors': ["Status must be either 'ON_ROUTE' or 'STOPPED'."], 'occupancy_category': None}


# Part D — Data Analysis

The dataset is analysed using Pandas.

The following metrics are calculated:
- average passenger count;
- maximum passenger count;
- number of STOPPED events;
- busiest minute;
- busiest bus.

In [26]:
def analyze_shuttle_data(
    events: List[Dict[str, Any]]
) -> Dict[str, Any]:

    df = pd.DataFrame(events)

    average_passengers = round(
        df["Passengers"].mean(), 2
    )

    maximum_passengers = int(
        df["Passengers"].max()
    )

    stopped_count = int(
        (df["Status"] == "STOPPED").sum()
    )

    busiest_idx = df["Passengers"].idxmax()
    busiest_row = df.loc[busiest_idx]

    return {
        "average_passengers": average_passengers,
        "max_passengers": maximum_passengers,
        "stopped_events_count": stopped_count,
        "busiest_bus": busiest_row["Bus"],
        "busiest_time": busiest_row["Timestamp"],
        "busiest_passenger_count": int(
            busiest_row["Passengers"]
        )
    }


metrics = analyze_shuttle_data(SAMPLE_DATA)

print("--- Analytics Results ---")

for key, value in metrics.items():
    print(f"{key}: {value}")

--- Analytics Results ---
average_passengers: 24.4
max_passengers: 30
stopped_events_count: 1
busiest_bus: B02
busiest_time: 08:09
busiest_passenger_count: 30


## Analysis Results

| Metric | Result |
|---|---:|
| Average passenger count | 24.4 |
| Maximum passenger count | 30 |
| Number of STOPPED events | 1 |
| Busiest minute | 08:09 |
| Busiest bus | B02 |
| Busiest passenger count | 30 |

# Part E — Testing and Code Quality

The implementation uses separate functions for validation, streaming, occupancy
classification and data analysis. This improves readability and makes individual
components easier to test.

Four test cases were used. The first three test normal validation and occupancy
behaviour, while the fourth is an additional boundary/validation test designed
specifically for this implementation.

| Test Case | Input | Expected Result | Actual Result | Status |
|---|---|---|---|---|
| Test 1 | 18 passengers, speed 31 | Accepted, MEDIUM | Accepted, MEDIUM | PASS |
| Test 2 | Speed 150 | Rejected | Rejected | PASS |
| Test 3 | 35 passengers | Accepted, OVER CAPACITY | Accepted, OVER CAPACITY | PASS |
| Test 4 | Status UNKNOWN | Rejected | Rejected | PASS |

# Technical Interpretation

Generators are preferable for unlimited data streams because they process events
one at a time instead of storing the entire stream in memory.

In this project, `event_stream_generator()` uses `yield` to provide events
sequentially. This approach is more memory-efficient and can be adapted to
continuous data sources.

One limitation of the current implementation is that the dataset is synthetic
and fixed. It does not receive data from a real-time external streaming system.

A possible improvement would be connecting the generator to a real streaming
source or message broker and adding logging, error handling and automatic
reconnection.

# 4. Individual Work and Anti-Plagiarism Record

### Exact source filename and functions/classes

This assignment was implemented in Google Colab in the notebook:

`Assignment_1_Asslan77.ipynb`

The main functions and class used in my implementation are:

- `validate_and_parse_event()` — validates and parses shuttle events.
- `get_occupancy_category()` — determines the passenger occupancy category.
- `event_stream_generator()` — generates shuttle events one at a time.
- `analyze_shuttle_data()` — calculates the main statistics.
- `ShuttleEventSchema` — Pydantic model used for FastAPI event validation.
- `process_event_endpoint()` — processes POST `/events` requests.

### Execution Evidence

All results reported in this assignment were generated by executing the Python cells in Google Colab.
The notebook contains the source code and the corresponding execution outputs.

### My Own Test Case

I added an additional test case for an invalid status value:

`Status = "UNKNOWN"`

According to the requirements, only `ON_ROUTE` and `STOPPED` are allowed.
Therefore, the expected result is that the event is rejected.

### My Implementation Decision

I separated the main functionality into independent functions for validation,
stream generation, occupancy classification and data analysis.

I selected this design because it makes the code easier to read, test and maintain.
It also allows each part of the program to be tested independently.

### Bug / Validation Problem

One validation issue I considered was that an event with an invalid status could otherwise
be processed as a normal event. I solved this by explicitly checking the Status field
against the allowed values `ON_ROUTE` and `STOPPED`.

I also validated the Speed_kmh field to make sure that values outside the 0–120 km/h range
are rejected.

### Limitation

The main limitation of my implementation is that the dataset is a fixed synthetic dataset.
It simulates streaming data but does not receive events from a real-time external source.

### Possible Improvement

A realistic improvement would be to connect the generator to a real-time streaming source
or message broker. Logging, error handling and automatic reconnection could also be added.

### Oral Defense

I am able to explain the implementation and reproduce or modify the validation, generator,
FastAPI endpoint and analysis functions during the oral defense.

# Conclusion

This assignment demonstrated the basic principles of Python-based streaming
data processing using a synthetic AITU campus shuttle dataset.

The implementation represents shuttle events using Python dictionaries,
validates incoming data, processes events using a generator, converts events
to JSON and provides a FastAPI REST endpoint.

The analysis showed an average passenger count of 24.4, a maximum of 30
passengers and one STOPPED event. The busiest event occurred at 08:09 on
bus B02 with 30 passengers.

The project also included four API test cases covering valid data, invalid
speed, over-capacity passengers and invalid status.

Generators provide an efficient approach for processing large or unlimited
streams because events can be processed sequentially without repeatedly
loading the complete stream into memory.